# 2.小文档块检索大文档块

知识点讲解：多级文档分割与父文档检索（小文档块检索大文档块）

1. 何时需要多级分割
   - 上一例把「父文档」设为完整原始文档，但有时完整文档太大，并不想按原样返回
   - 此时希望：先把原始文档切成较大的块（如 1000~2000 字符/token），再切成小块索引，检索时返回较大块而非原文档
   - 这样在检索精度和返回长度之间取得更好的平衡，也节省 token

2. 核心概念：三级结构
   - 原始文档 → 父文档块（parent_splitter 切出的大块）→ 子文档块（child_splitter 切出的小块）
   - parent_splitter：生成较大的文档块，作为最终返回内容
   - child_splitter：生成较小的文档块，用于生成嵌入并检索
   - 分层存储：向量库存小块，文档库存中块（父文档块）

3. 组件定位与复用
   - 与基本用法共用同一个 ParentDocumentRetriever，无需改动检索流程
   - 唯一区别：多传一个 parent_splitter 参数，框架据此把「父文档」从「完整原文档」换成「大块」
   - 其余（vectorstore、byte_store、child_splitter）完全不变，体现了对开闭原则的践行

4. 本示例配置（小块检索，中块返回）
   - parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)：父文档块约 2000 字符，存入文档数据库
   - child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)：子文档块约 500 字符，用于检索
   - 检索时用小块匹配查询（精确），命中后返回对应的 2000 字符大块（足够上下文）

5. 与基本用法的对比
   - 基本用法：不设置 parent_splitter，返回完整原始文档
   - 高级用法：设置 parent_splitter，返回中等大小的文档块
   - 取舍：高级用法避免返回过长文档（节省 token），同时保持足够上下文，适合超长文档

6. 优势与适用场景
   - 更灵活的粒度控制；避免返回过长文档（省 token）；保持足够上下文；适合超长文档
   - 适用：书籍、长篇文章等超长文档；需精确控制返回内容长度；多章节文档的分段检索

7. 工作原理（五个步骤）
   - 用 parent_splitter 将原始文档切成大块（如 2000 字符）
   - 用 child_splitter 将每个大块进一步切成小块（如 500 字符）
   - 小块生成嵌入向量存入向量数据库
   - 大块（父文档）存入文档数据库
   - 检索用小块匹配（精确），返回对应大块（足够上下文）

8. 最佳实践
   - parent_splitter 大小建议 1500~3000 字符（提供足够上下文）
   - child_splitter 大小建议 300~600 字符（保证检索精度）
   - 保持合理的 chunk_overlap 避免语义断裂
   - 根据具体文档类型调整分割策略


In [ ]:
import dotenv
import weaviate
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore
from langchain_unstructured import UnstructuredLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_weaviate import WeaviateVectorStore
from weaviate.auth import AuthApiKey
dotenv.load_dotenv()
loaders = [
    UnstructuredLoader("./电商产品数据.txt"),
    UnstructuredLoader("./项目API文档.md"),
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [ ]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
vector_store = WeaviateVectorStore(
    client=weaviate.connect_to_wcs(
        cluster_url="https://mbakeruerziae6psyex7ng.c0.us-west3.gcp.weaviate.cloud",
        auth_credentials=AuthApiKey("ZltPVa9ZSOxUcfafelsggGyyH6tnTYQYJvBx"),
    ),
    index_name="ParentDocument",
    text_key="text",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)
store = LocalFileStore("./parent-document")
retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    byte_store=store,
    parent_splitter=parent_splitter,
    child_splitter=child_splitter,
)
retriever.add_documents(docs, ids=None)
search_docs = retriever.invoke("分享关于LLMOps的一些应用配置")
print(search_docs)
print(len(search_docs))